In [1]:
# Celda 1 — verificar entorno
import sys
sys.path.append("..")

import chromadb
from sentence_transformers import SentenceTransformer
import httpx

print("✅ Librerías cargadas correctamente")
print(f"Python: {sys.version[:6]}")

C:\proyectos\clean_standarization_RAG\.venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


✅ Librerías cargadas correctamente
Python: 3.12.5


In [2]:
# Celda 2 — conectar ChromaDB
import os
from dotenv import load_dotenv

load_dotenv("../.env")

chroma_path = os.getenv("CHROMA_PATH", "./data/chroma")
client = chromadb.PersistentClient(path=f"../{chroma_path.lstrip('./')}")

# Crear colección de prueba
coleccion = client.get_or_create_collection(
    name="prueba_rag",
    metadata={"descripcion": "colección temporal para probar el pipeline RAG"}
)

print("✅ ChromaDB conectado")
print(f"Colección: {coleccion.name}")
print(f"Documentos actuales: {coleccion.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


✅ ChromaDB conectado
Colección: prueba_rag
Documentos actuales: 0


In [3]:
# Celda 3 — cargar modelo de embeddings
modelo = SentenceTransformer("paraphrase-multilingual-mpnet-base-v2")
print("✅ Modelo cargado")
print(f"Dimensiones: {modelo.encode('prueba').shape[0]}")

✅ Modelo cargado
Dimensiones: 768


In [4]:
# Celda 4 — insertar chunks de prueba
# Estos son chunks inventados que simulan lo que generaría el pipeline real

chunks = [
    "En la encuesta de hábitos de estudio aplicada a 120 estudiantes de ESCOM en marzo de 2024, el 75% reportó estudiar menos de 2 horas diarias, lo que indica un déficit en el KPI de tiempo dedicado al estudio.",
    "En la encuesta de hábitos de estudio de ESCOM 2024, el 60% de los 120 estudiantes encuestados prefiere clases presenciales sobre virtuales, especialmente en materias de programación.",
    "En la encuesta aplicada a 50 estudiantes de IIA en febrero de 2024, el 80% indicó que los recursos del laboratorio son insuficientes para cubrir las prácticas del semestre.",
    "En la encuesta de satisfacción escolar de ESCOM 2024, el 45% de 120 estudiantes calificó la atención administrativa con menos de 6 sobre 10, siendo el tiempo de espera el principal problema.",
    "En la encuesta de hábitos alimenticios aplicada a 90 estudiantes de ESCOM en enero de 2024, el 65% no desayuna antes de entrar a clases, lo que correlaciona con bajo rendimiento en materias matutinas.",
]

# Generar embeddings para cada chunk
print("Generando embeddings...")
vectores = modelo.encode(chunks)
print(f"✅ {len(vectores)} vectores generados de {vectores[0].shape[0]} dims cada uno")

# Insertar en ChromaDB
coleccion.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[v.tolist() for v in vectores],
    metadatas=[
        {"tipo": "prueba", "instrumento": "encuesta", "indice": i}
        for i in range(len(chunks))
    ]
)

print(f"✅ Chunks insertados en ChromaDB")
print(f"Total en colección: {coleccion.count()}")

Generando embeddings...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


✅ 5 vectores generados de 768 dims cada uno
✅ Chunks insertados en ChromaDB
Total en colección: 5


In [6]:
# Celda 5 — inspeccionar ChromaDB
import numpy as np

todos = coleccion.get(include=["documents", "metadatas", "embeddings"])

print(f"{'═'*60}")
print(f"  CONTENIDO DE CHROMADB — colección: {coleccion.name}")
print(f"{'═'*60}")
print(f"  Total chunks: {len(todos['ids'])}")
print()

for i, (id_, doc, meta, emb) in enumerate(zip(
    todos["ids"],
    todos["documents"],
    todos["metadatas"],
    todos["embeddings"]
)):
    vector = np.array(emb)
    # Estimación de tokens — aproximación: 1 token ≈ 4 caracteres en español
    tokens_estimados = len(doc) // 4
    palabras = len(doc.split())
    
    print(f"┌─ {id_} ─────────────────────────────────────")
    print(f"│  Texto:          {doc[:60]}...")
    print(f"│  Palabras:       {palabras}")
    print(f"│  Tokens aprox:   {tokens_estimados}")
    print(f"│  Dimensiones:    {vector.shape[0]}")
    print(f"│  Vector min/max: {vector.min():.4f} / {vector.max():.4f}")
    print(f"│  Metadata:       {meta}")
    print(f"└{'─'*50}")
    print()

════════════════════════════════════════════════════════════
  CONTENIDO DE CHROMADB — colección: prueba_rag
════════════════════════════════════════════════════════════
  Total chunks: 5

┌─ chunk_0 ─────────────────────────────────────
│  Texto:          En la encuesta de hábitos de estudio aplicada a 120 estudian...
│  Palabras:       39
│  Tokens aprox:   51
│  Dimensiones:    768
│  Vector min/max: -0.3756 / 0.4029
│  Metadata:       {'indice': 0, 'instrumento': 'encuesta', 'tipo': 'prueba'}
└──────────────────────────────────────────────────

┌─ chunk_1 ─────────────────────────────────────
│  Texto:          En la encuesta de hábitos de estudio de ESCOM 2024, el 60% d...
│  Palabras:       27
│  Tokens aprox:   45
│  Dimensiones:    768
│  Vector min/max: -0.3833 / 0.3982
│  Metadata:       {'indice': 1, 'instrumento': 'encuesta', 'tipo': 'prueba'}
└──────────────────────────────────────────────────

┌─ chunk_2 ─────────────────────────────────────
│  Texto:          En la encue

In [7]:
# Celda 6 — consulta RAG completa
query = "¿Cuánto estudian los estudiantes de ESCOM?"

print(f"{'═'*60}")
print(f"  CONSULTA RAG")
print(f"{'═'*60}")
print(f"  Query: '{query}'")

# Paso 1 — convertir query a vector
vector_query = modelo.encode(query).tolist()
print(f"\n Paso 1 — Query convertida a vector de {len(vector_query)} dims")

# Paso 2 — buscar chunks similares en ChromaDB
resultados = coleccion.query(
    query_embeddings=[vector_query],
    n_results=2
)

print(f"\n Paso 2 — Chunks recuperados:")
contexto = ""
for i, (doc, distancia) in enumerate(zip(
    resultados["documents"][0],
    resultados["distances"][0]
)):
    print(f"\n  Chunk {i+1} (distancia: {distancia:.4f})")
    print(f"  {doc}")
    contexto += f"\n{doc}"

# Paso 3 — LLM genera respuesta con los chunks como contexto
print(f"\n Paso 3 — Enviando al LLM...")

respuesta = httpx.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "llama3.2:3b",
        "prompt": f"""Eres un asistente de análisis de datos educativos.
Responde la pregunta del investigador basándote ÚNICAMENTE en la información 
de los siguientes fragmentos de encuestas. Si la información no está en los 
fragmentos, dilo explícitamente.

FRAGMENTOS:
{contexto}

PREGUNTA: {query}

RESPUESTA:""",
        "stream": False,
        "options": {"temperature": 0.3}
    },
    timeout=120.0
)

respuesta_llm = respuesta.json()["response"]

print(f"\n{'═'*60}")
print(f"  RESPUESTA DEL LLM")
print(f"{'═'*60}")
print(f"\n{respuesta_llm}")

════════════════════════════════════════════════════════════
  CONSULTA RAG
════════════════════════════════════════════════════════════
  Query: '¿Cuánto estudian los estudiantes de ESCOM?'


Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



 Paso 1 — Query convertida a vector de 768 dims

 Paso 2 — Chunks recuperados:

  Chunk 1 (distancia: 5.4790)
  En la encuesta de hábitos de estudio aplicada a 120 estudiantes de ESCOM en marzo de 2024, el 75% reportó estudiar menos de 2 horas diarias, lo que indica un déficit en el KPI de tiempo dedicado al estudio.

  Chunk 2 (distancia: 7.0541)
  En la encuesta de satisfacción escolar de ESCOM 2024, el 45% de 120 estudiantes calificó la atención administrativa con menos de 6 sobre 10, siendo el tiempo de espera el principal problema.

 Paso 3 — Enviando al LLM...

════════════════════════════════════════════════════════════
  RESPUESTA DEL LLM
════════════════════════════════════════════════════════════

Basándome únicamente en los fragmentos de encuesta proporcionados, no puedo determinar la cantidad exacta de horas que estudian los estudiantes de ESCOM. El fragmento de encuesta sobre hábitos de estudio menciona que el 75% de los estudiantes reportó estudiar menos de 2 horas diari

In [8]:
# Celda 7 — misma query pero con mistral
print(f"{'═'*60}")
print(f"  MISMA CONSULTA — modelo: mistral:latest")
print(f"{'═'*60}")
print(f"  Query: '¿Cuánto estudian los estudiantes de ESCOM?'")

# Los chunks ya los tenemos de la celda anterior
# solo cambiamos el modelo
respuesta_mistral = httpx.post(
    "http://localhost:11434/api/generate",
    json={
        "model": "mistral:latest",
        "prompt": f"""Eres un asistente de análisis de datos educativos.
Responde la pregunta del investigador basándote ÚNICAMENTE en la información 
de los siguientes fragmentos de encuestas. Si la información no está en los 
fragmentos, dilo explícitamente.

FRAGMENTOS:
{contexto}

PREGUNTA: ¿Cuánto estudian los estudiantes de ESCOM?

RESPUESTA:""",
        "stream": False,
        "options": {"temperature": 0.3}
    },
    timeout=120.0
)

respuesta_mistral_txt = respuesta_mistral.json()["response"]

print(f"\n{respuesta_mistral_txt}")

# Comparación directa
print(f"\n{'═'*60}")
print(f"  COMPARACIÓN")
print(f"{'═'*60}")
print(f"\n  llama3.2:3b ({len(respuesta_llm.split())} palabras):")
print(f"  {respuesta_llm[:150]}...")
print(f"\n  mistral:latest ({len(respuesta_mistral_txt.split())} palabras):")
print(f"  {respuesta_mistral_txt[:150]}...")

════════════════════════════════════════════════════════════
  MISMA CONSULTA — modelo: mistral:latest
════════════════════════════════════════════════════════════
  Query: '¿Cuánto estudian los estudiantes de ESCOM?'

 Los datos proporcionados indican que el 75% de los estudiantes de ESCOM estudian menos de 2 horas diarias. Sin embargo, no se proporciona información sobre la cantidad de tiempo estudiado por el resto del 25%. Por lo tanto, es difícil determinar cuánto estudian los estudiantes de ESCOM en general basándome únicamente en los fragmentos de encuestas proporcionados.

════════════════════════════════════════════════════════════
  COMPARACIÓN
════════════════════════════════════════════════════════════

  llama3.2:3b (101 palabras):
  Basándome únicamente en los fragmentos de encuesta proporcionados, no puedo determinar la cantidad exacta de horas que estudian los estudiantes de ESC...

  mistral:latest (57 palabras):
   Los datos proporcionados indican que el 75% de los est

In [9]:
# Celda 8 — nueva colección con similitud coseno
coleccion_coseno = client.get_or_create_collection(
    name="prueba_rag_coseno",
    metadata={"hnsw:space": "cosine"}
)

# Insertar los mismos chunks
coleccion_coseno.add(
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    documents=chunks,
    embeddings=[v.tolist() for v in vectores],
    metadatas=[
        {"tipo": "prueba", "instrumento": "encuesta", "indice": i}
        for i in range(len(chunks))
    ]
)

print(" Colección con similitud coseno creada")
print(f"Total chunks: {coleccion_coseno.count()}")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


 Colección con similitud coseno creada
Total chunks: 5


In [10]:
# Celda 9 — consulta con similitud coseno
query = "¿Cuánto estudian los estudiantes de ESCOM?"
vector_query = modelo.encode(query).tolist()

resultados_coseno = coleccion_coseno.query(
    query_embeddings=[vector_query],
    n_results=2
)

print(f"{'═'*60}")
print(f"  COMPARACIÓN DE MÉTRICAS DE SIMILITUD")
print(f"{'═'*60}")

print(f"\n  Query: '{query}'")

print(f"\n  {'─'*55}")
print(f"  DISTANCIA EUCLIDIANA (colección original)")
print(f"  {'─'*55}")
for i, (doc, dist) in enumerate(zip(
    resultados["documents"][0],
    resultados["distances"][0]
)):
    print(f"  Chunk {i+1}: {dist:.4f} → {doc[:60]}...")

print(f"\n  {'─'*55}")
print(f"  SIMILITUD COSENO (nueva colección)")
print(f"  {'─'*55}")
for i, (doc, dist) in enumerate(zip(
    resultados_coseno["documents"][0],
    resultados_coseno["distances"][0]
)):
    # Con coseno: 0 = idéntico, 1 = opuesto
    similitud = 1 - dist
    print(f"  Chunk {i+1}: distancia={dist:.4f} → similitud={similitud:.4f} → {doc[:60]}...")

print(f"\n  {'─'*55}")
print(f"  INTERPRETACIÓN")
print(f"  {'─'*55}")
print(f"  Euclidiana: menor número = más similar")
print(f"  Coseno:     similitud más cercana a 1 = más similar")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


════════════════════════════════════════════════════════════
  COMPARACIÓN DE MÉTRICAS DE SIMILITUD
════════════════════════════════════════════════════════════

  Query: '¿Cuánto estudian los estudiantes de ESCOM?'

  ───────────────────────────────────────────────────────
  DISTANCIA EUCLIDIANA (colección original)
  ───────────────────────────────────────────────────────
  Chunk 1: 5.4790 → En la encuesta de hábitos de estudio aplicada a 120 estudian...
  Chunk 2: 7.0541 → En la encuesta de satisfacción escolar de ESCOM 2024, el 45%...

  ───────────────────────────────────────────────────────
  SIMILITUD COSENO (nueva colección)
  ───────────────────────────────────────────────────────
  Chunk 1: distancia=0.3499 → similitud=0.6501 → En la encuesta de hábitos de estudio aplicada a 120 estudian...
  Chunk 2: distancia=0.4596 → similitud=0.5404 → En la encuesta de satisfacción escolar de ESCOM 2024, el 45%...

  ───────────────────────────────────────────────────────
  INTERPRETACIÓN

In [11]:
# Celda 10 — colección con metadatos ricos
coleccion_meta = client.get_or_create_collection(
    name="prueba_rag_metadatos",
    metadata={"hnsw:space": "cosine"}
)

chunks_ricos = [
    {
        "texto": "En la encuesta de hábitos de estudio aplicada a 120 estudiantes de ESCOM en marzo de 2024, el 75% reportó estudiar menos de 2 horas diarias, lo que indica un déficit en el KPI de tiempo dedicado al estudio.",
        "metadata": {
            "tipo_instrumento": "encuesta",
            "institucion": "ESCOM",
            "carrera": "ISC",
            "anio": "2024",
            "tema": "habitos_estudio",
            "kpi": "tiempo_estudio",
            "n_respondentes": 120
        }
    },
    {
        "texto": "En la encuesta de hábitos de estudio de ESCOM 2024, el 60% de los 120 estudiantes prefiere clases presenciales sobre virtuales, especialmente en materias de programación.",
        "metadata": {
            "tipo_instrumento": "encuesta",
            "institucion": "ESCOM",
            "carrera": "ISC",
            "anio": "2024",
            "tema": "modalidad_clases",
            "kpi": "preferencia_modalidad",
            "n_respondentes": 120
        }
    },
    {
        "texto": "En la encuesta aplicada a 50 estudiantes de IIA en febrero de 2024, el 80% indicó que los recursos del laboratorio son insuficientes para cubrir las prácticas del semestre.",
        "metadata": {
            "tipo_instrumento": "encuesta",
            "institucion": "ESCOM",
            "carrera": "IIA",
            "anio": "2024",
            "tema": "recursos_laboratorio",
            "kpi": "satisfaccion_infraestructura",
            "n_respondentes": 50
        }
    },
    {
        "texto": "En la entrevista estructurada aplicada a 10 profesores de ESCOM en abril de 2024, el 90% mencionó que el tiempo de clase es insuficiente para cubrir el temario completo.",
        "metadata": {
            "tipo_instrumento": "entrevista",
            "institucion": "ESCOM",
            "carrera": "todas",
            "anio": "2024",
            "tema": "tiempo_clase",
            "kpi": "cobertura_temario",
            "n_respondentes": 10
        }
    },
    {
        "texto": "En la prueba estandarizada aplicada a 200 estudiantes de ISC de ESCOM en mayo de 2024, el promedio general fue de 6.8 sobre 10, con mayor déficit en álgebra lineal.",
        "metadata": {
            "tipo_instrumento": "prueba_estandarizada",
            "institucion": "ESCOM",
            "carrera": "ISC",
            "anio": "2024",
            "tema": "rendimiento_academico",
            "kpi": "promedio_general",
            "n_respondentes": 200
        }
    },
]

# Insertar
vectores_ricos = modelo.encode([c["texto"] for c in chunks_ricos])

coleccion_meta.add(
    ids=[f"chunk_{i}" for i in range(len(chunks_ricos))],
    documents=[c["texto"] for c in chunks_ricos],
    embeddings=[v.tolist() for v in vectores_ricos],
    metadatas=[c["metadata"] for c in chunks_ricos]
)

print(" Colección con metadatos ricos creada")
print(f"Total chunks: {coleccion_meta.count()}")
print("\nInstrumentos incluidos:")
print("  - 3 encuestas")
print("  - 1 entrevista")
print("  - 1 prueba estandarizada")

Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


 Colección con metadatos ricos creada
Total chunks: 5

Instrumentos incluidos:
  - 3 encuestas
  - 1 entrevista
  - 1 prueba estandarizada


In [12]:
# Celda 11 — búsqueda filtrada por metadatos
query = "¿Cuáles son los problemas que enfrentan los estudiantes de ESCOM?"
vector_query = modelo.encode(query).tolist()

print(f"{'═'*60}")
print(f"  BÚSQUEDA FILTRADA POR METADATOS")
print(f"{'═'*60}")
print(f"  Query: '{query}'")

# ── Búsqueda 1: SIN filtro — busca en todo ────────────────
print(f"\n{'─'*55}")
print(f"  SIN FILTRO — todos los instrumentos")
print(f"{'─'*55}")
sin_filtro = coleccion_meta.query(
    query_embeddings=[vector_query],
    n_results=3
)
for i, (doc, meta, dist) in enumerate(zip(
    sin_filtro["documents"][0],
    sin_filtro["metadatas"][0],
    sin_filtro["distances"][0]
)):
    similitud = round(1 - dist, 4)
    print(f"\n  Chunk {i+1} — similitud: {similitud}")
    print(f"  Tipo: {meta['tipo_instrumento']} | Carrera: {meta['carrera']}")
    print(f"  {doc[:80]}...")

# ── Búsqueda 2: SOLO encuestas ────────────────────────────
print(f"\n{'─'*55}")
print(f"  CON FILTRO — solo encuestas")
print(f"{'─'*55}")
solo_encuestas = coleccion_meta.query(
    query_embeddings=[vector_query],
    n_results=3,
    where={"tipo_instrumento": "encuesta"}
)
for i, (doc, meta, dist) in enumerate(zip(
    solo_encuestas["documents"][0],
    solo_encuestas["metadatas"][0],
    solo_encuestas["distances"][0]
)):
    similitud = round(1 - dist, 4)
    print(f"\n  Chunk {i+1} — similitud: {similitud}")
    print(f"  Tipo: {meta['tipo_instrumento']} | Carrera: {meta['carrera']}")
    print(f"  {doc[:80]}...")

# ── Búsqueda 3: SOLO carrera ISC ─────────────────────────
print(f"\n{'─'*55}")
print(f"  CON FILTRO — solo carrera ISC")
print(f"{'─'*55}")
solo_isc = coleccion_meta.query(
    query_embeddings=[vector_query],
    n_results=3,
    where={"carrera": "ISC"}
)
for i, (doc, meta, dist) in enumerate(zip(
    solo_isc["documents"][0],
    solo_isc["metadatas"][0],
    solo_isc["distances"][0]
)):
    similitud = round(1 - dist, 4)
    print(f"\n  Chunk {i+1} — similitud: {similitud}")
    print(f"  Tipo: {meta['tipo_instrumento']} | Carrera: {meta['carrera']}")
    print(f"  {doc[:80]}...")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


════════════════════════════════════════════════════════════
  BÚSQUEDA FILTRADA POR METADATOS
════════════════════════════════════════════════════════════
  Query: '¿Cuáles son los problemas que enfrentan los estudiantes de ESCOM?'

───────────────────────────────────────────────────────
  SIN FILTRO — todos los instrumentos
───────────────────────────────────────────────────────

  Chunk 1 — similitud: 0.5484
  Tipo: entrevista | Carrera: todas
  En la entrevista estructurada aplicada a 10 profesores de ESCOM en abril de 2024...

  Chunk 2 — similitud: 0.5235
  Tipo: prueba_estandarizada | Carrera: ISC
  En la prueba estandarizada aplicada a 200 estudiantes de ISC de ESCOM en mayo de...

  Chunk 3 — similitud: 0.4356
  Tipo: encuesta | Carrera: ISC
  En la encuesta de hábitos de estudio de ESCOM 2024, el 60% de los 120 estudiante...

───────────────────────────────────────────────────────
  CON FILTRO — solo encuestas
───────────────────────────────────────────────────────

  Chunk 1